# Day 6 — Topic A: LiDAR-camera calibration QA
**Nguyễn Văn Chiến · 2A202602926 · AI20K**

Trong Colab chọn **Runtime → Run all**. CPU đủ cho bài này, không cần GPU hoặc Google Drive.
Notebook tải repo khoảng 135 MB, mang sẵn code projection/benchmark đã hoàn thiện và tạo lại CSV, ảnh, báo cáo từ dữ liệu thật sự chạy trong phiên này. Không cần push các thay đổi cục bộ lên GitHub trước.
Cell cuối tải một ZIP gồm code, kết quả và báo cáo; dữ liệu gốc không nằm trong ZIP.


In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util

REPO_URL = 'https://github.com/nvchien19/K4-Track4-Day06-3D-From-Point-Clouds.git'
BASE_COMMIT = 'bce73adec3dbd09b2869ed2061513328ee228272'
work = Path('/content/day6-topic-a-' + BASE_COMMIT[:8])
if not work.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(work)], check=True)
    subprocess.run(['git', '-C', str(work), 'checkout', '--detach', BASE_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(work), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == BASE_COMMIT, 'Thư mục đã có nhưng commit khác: hãy dùng phiên Colab mới.'
os.chdir(work)
if str(work) not in sys.path:
    sys.path.insert(0, str(work))
packages = {'numpy': 'numpy>=1.24', 'cv2': 'opencv-python-headless>=4.8', 'matplotlib': 'matplotlib>=3.7'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('Workspace:', work, '\nBase commit:', actual)


## 1. Khôi phục code hoàn thiện
Cell này ghi code được mang theo trong notebook, không phụ thuộc vào các TODO trên GitHub.

In [ ]:
PAYLOAD = {'starter/projection.py': '"""LiDAR -> camera projection, overlay và perturb calibration.\n\nHai hàm có `TODO(CP2)` là phần học viên phải tự cài đặt (bắt buộc cho topic A, C, F;\nkhuyến khích cho mọi topic vì đây là bài test calibration rẻ nhất).\n\nChạy thử sau khi cài đặt xong (cùng một code chạy được cho cả KITTI và nuScenes):\n    python -m starter.projection --data-root data/synthetic --frame 000000\n    python -m starter.projection --data-root data/synthetic --frame 000000 --yaw-deg 1.0\n    python -m starter.projection --data-root data/kitti_mini --frame 000011\n    python -m starter.projection --data-root data/nuscenes_mini_subset --frame scene-0103_010\n    python -m starter.projection --data-root data/nuscenes_mini_subset --frame scene-0103_010 --ignore-ego-motion\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom starter.datasets import dataset_type, load_frame\nfrom starter.kitti_io import KittiCalib, KittiObject\n\n\ndef velo_to_cam(points_xyz: np.ndarray, calib: KittiCalib) -> np.ndarray:\n    """Đưa điểm (N, 3) từ velodyne frame sang rectified camera frame (N, 3).\n\n    TODO(CP2):\n      1. Chuyển sang toạ độ đồng nhất (N, 4).\n      2. Nhân với calib.T_cam_velo (4x4). Chú ý chiều nhân và transpose.\n      3. Trả về 3 cột đầu.\n    Tự kiểm: một điểm velodyne (10, 0, 0) phải có z_cam ~ 10 (phía trước camera).\n    """\n    points_xyz = np.asarray(points_xyz, dtype=np.float64)\n    if points_xyz.ndim != 2 or points_xyz.shape[1] != 3:\n        raise ValueError("points_xyz phải có shape (N, 3)")\n    points_h = np.concatenate(\n        [points_xyz, np.ones((len(points_xyz), 1), dtype=points_xyz.dtype)], axis=1\n    )\n    return (calib.T_cam_velo @ points_h.T).T[:, :3]\n\n\ndef cam_to_image(points_cam: np.ndarray, P2: np.ndarray, image_shape: tuple[int, ...],\n                 min_depth: float = 0.1) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    """Chiếu điểm camera frame (N, 3) lên ảnh bằng P2 (3x4).\n\n    Trả về:\n      uv    (M, 2) toạ độ pixel của các điểm hợp lệ\n      depth (M,)   z_cam của các điểm hợp lệ\n      mask  (N,)   bool, True nếu điểm hợp lệ\n\n    Điểm hợp lệ = depth > min_depth VÀ nằm trong ảnh (0 <= u < W, 0 <= v < H).\n\n    TODO(CP2):\n      1. Lọc điểm không hợp lệ (NaN/Inf): dữ liệu thật không bao giờ sạch.\n      2. Toạ độ đồng nhất, nhân P2 -> (N, 3) = [s*u, s*v, s].\n      3. Chia cho s để có (u, v). Chỉ chia với điểm có depth > min_depth.\n      4. Lọc theo kích thước ảnh image_shape[:2] = (H, W).\n    """\n    points_cam = np.asarray(points_cam, dtype=np.float64)\n    if points_cam.ndim != 2 or points_cam.shape[1] != 3:\n        raise ValueError("points_cam phải có shape (N, 3)")\n\n    height, width = image_shape[:2]\n    finite = np.isfinite(points_cam).all(axis=1)\n    depth_all = points_cam[:, 2]\n    depth_ok = finite & (depth_all > min_depth)\n\n    # Avoid propagating NaN/Inf into matrix multiplication; retain original row\n    # alignment so the returned mask always has length N.\n    uv_all = np.full((len(points_cam), 2), np.nan, dtype=np.float64)\n    idx = np.flatnonzero(depth_ok)\n    if idx.size:\n        points_h = np.column_stack([points_cam[idx], np.ones(idx.size)])\n        projected = (np.asarray(P2, dtype=np.float64) @ points_h.T).T\n        scale = projected[:, 2]\n        projectable = np.isfinite(projected).all(axis=1) & (np.abs(scale) > 1e-12)\n        uv_all[idx[projectable]] = projected[projectable, :2] / scale[projectable, None]\n\n    inside = (\n        depth_ok\n        & np.isfinite(uv_all).all(axis=1)\n        & (uv_all[:, 0] >= 0)\n        & (uv_all[:, 0] < width)\n        & (uv_all[:, 1] >= 0)\n        & (uv_all[:, 1] < height)\n    )\n    return uv_all[inside], depth_all[inside], inside\n\n\ndef project_velo_to_image(points: np.ndarray, calib: KittiCalib, image_shape: tuple[int, ...]):\n    """points (N, >=3) velodyne -> (uv, depth, mask) như `cam_to_image`."""\n    return cam_to_image(velo_to_cam(points[:, :3], calib), calib.P2, image_shape)\n\n\ndef overlay_points(image: np.ndarray, uv: np.ndarray, depth: np.ndarray,\n                   max_depth: float = 50.0, radius: int = 2) -> np.ndarray:\n    """Vẽ điểm lên ảnh, màu theo depth (gần = đỏ, xa = xanh)."""\n    out = image.copy()\n    d = np.clip(depth / max_depth, 0, 1)\n    colors = cv2.applyColorMap((255 * (1 - d)).astype(np.uint8).reshape(-1, 1), cv2.COLORMAP_JET)\n    for (u, v), c in zip(uv.astype(int), colors[:, 0]):\n        cv2.circle(out, (int(u), int(v)), radius, tuple(int(x) for x in c), -1)\n    return out\n\n\ndef _rot(roll: float, pitch: float, yaw: float) -> np.ndarray:\n    cr, sr, cp, sp, cy, sy = np.cos(roll), np.sin(roll), np.cos(pitch), np.sin(pitch), np.cos(yaw), np.sin(yaw)\n    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])\n    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])\n    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])\n    return Rz @ Ry @ Rx\n\n\ndef perturb_extrinsic(calib: KittiCalib, roll_deg: float = 0.0, pitch_deg: float = 0.0,\n                      yaw_deg: float = 0.0, t_xyz_m: tuple[float, float, float] = (0, 0, 0)) -> KittiCalib:\n    """Giả lập calibration drift: xoay/dịch LiDAR trong chính velodyne frame\n    (yaw quanh z-up, pitch quanh y-left, roll quanh x-forward) rồi ghép vào Tr_velo_to_cam."""\n    D = np.eye(4)\n    D[:3, :3] = _rot(*np.deg2rad([roll_deg, pitch_deg, yaw_deg]))\n    D[:3, 3] = t_xyz_m\n    Tr = np.eye(4)\n    Tr[:3, :] = calib.Tr_velo_to_cam\n    out = copy.deepcopy(calib)\n    out.Tr_velo_to_cam = (Tr @ D)[:3, :]\n    return out\n\n\ndef box3d_corners_cam(obj: KittiObject) -> np.ndarray:\n    """8 góc (8, 3) của box KITTI trong rectified camera frame.\n    Thứ tự: 4 góc đáy (y=0) rồi 4 góc nóc (y=-h)."""\n    h, w, l = obj.dimensions\n    x = np.array([l, l, -l, -l, l, l, -l, -l]) / 2\n    y = np.array([0, 0, 0, 0, -h, -h, -h, -h])\n    z = np.array([w, -w, -w, w, w, -w, -w, w]) / 2\n    c, s = np.cos(obj.rotation_y), np.sin(obj.rotation_y)\n    R = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])\n    return (R @ np.vstack([x, y, z])).T + obj.location\n\n\ndef draw_box2d(image: np.ndarray, bbox, color=(0, 255, 0), label: str | None = None) -> np.ndarray:\n    out = image.copy()\n    x1, y1, x2, y2 = (int(round(v)) for v in bbox)\n    cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)\n    if label:\n        cv2.putText(out, label, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)\n    return out\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description="Chiếu điểm LiDAR lên ảnh camera, tô màu theo độ sâu, vẽ 2D box của label")\n    ap.add_argument("--data-root", default="data/synthetic",\n                    help="thư mục KITTI (data/synthetic, data/kitti_mini) hoặc nuScenes (data/nuscenes_mini_subset)")\n    ap.add_argument("--frame", default="000000", help="KITTI: 000011. nuScenes: scene-0103_010")\n    ap.add_argument("--out-dir", default="results/figures", help="nơi lưu ảnh overlay")\n    ap.add_argument("--roll-deg", type=float, default=0.0, help="giả lập LiDAR bị xoay quanh trục x (độ)")\n    ap.add_argument("--pitch-deg", type=float, default=0.0, help="giả lập LiDAR bị xoay quanh trục y (độ)")\n    ap.add_argument("--yaw-deg", type=float, default=0.0, help="giả lập LiDAR bị xoay quanh trục z hướng lên (độ)")\n    ap.add_argument("--tx", type=float, default=0.0, help="giả lập LiDAR bị dịch theo trục x của LiDAR (mét)")\n    ap.add_argument("--ty", type=float, default=0.0, help="giả lập LiDAR bị dịch theo trục y của LiDAR (mét)")\n    ap.add_argument("--tz", type=float, default=0.0, help="giả lập LiDAR bị dịch theo trục z của LiDAR (mét)")\n    ap.add_argument("--ignore-ego-motion", action="store_true",\n                    help="chỉ cho nuScenes: bỏ bù chuyển động xe giữa thời điểm chụp LiDAR và camera")\n    args = ap.parse_args()\n\n    kwargs = {"use_ego_motion": not args.ignore_ego_motion} if dataset_type(args.data_root) == "nuscenes" else {}\n    fr = load_frame(args.data_root, args.frame, **kwargs)\n    calib = perturb_extrinsic(fr["calib"], args.roll_deg, args.pitch_deg, args.yaw_deg,\n                              (args.tx, args.ty, args.tz))\n    uv, depth, mask = project_velo_to_image(fr["points"], calib, fr["image"].shape)\n    vis = overlay_points(fr["image"], uv, depth)\n    for obj in fr["labels"]:\n        vis = draw_box2d(vis, obj.bbox, label=obj.type)\n\n    tag = f"r{args.roll_deg}_p{args.pitch_deg}_y{args.yaw_deg}_t{args.tx}_{args.ty}_{args.tz}"\n    if args.ignore_ego_motion:\n        tag += "_noego"\n    out = Path(args.out_dir) / f"overlay_{args.frame}_{tag}.png"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    cv2.imwrite(str(out), vis)\n    print(f"points={len(mask)} inside_image={int(mask.sum())} ({mask.mean():.1%}) -> {out}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/topic_a_calibration.py': '"""Topic A: measure how yaw calibration drift changes LiDAR-to-camera overlays."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nfrom pathlib import Path\n\nimport cv2\nimport matplotlib.pyplot as plt\nimport numpy as np\n\nfrom starter.datasets import load_frame\nfrom starter.projection import overlay_points, perturb_extrinsic, project_velo_to_image\n\n\ndef _box_membership(uv: np.ndarray, labels) -> tuple[np.ndarray, list[dict]]:\n    """Return unique projected points inside any labeled 2D box and per-object counts."""\n    all_members = np.zeros(len(uv), dtype=bool)\n    objects = []\n    for index, obj in enumerate(labels):\n        x1, y1, x2, y2 = obj.bbox\n        inside = (\n            (uv[:, 0] >= x1) & (uv[:, 0] <= x2)\n            & (uv[:, 1] >= y1) & (uv[:, 1] <= y2)\n        )\n        all_members |= inside\n        objects.append({\n            "object_index": index,\n            "class": obj.type,\n            "points_in_box": int(inside.sum()),\n            "box_area_px": float(max(0, x2 - x1) * max(0, y2 - y1)),\n        })\n    return all_members, objects\n\n\ndef run(data_root: str, frame_id: str, yaw_degs: list[float], out_dir: Path) -> list[dict]:\n    frame = load_frame(data_root, frame_id)\n    image = frame["image"]\n    raw_points = np.asarray(frame["points"])\n    finite_xyz = np.isfinite(raw_points[:, :3]).all(axis=1)\n    rows = []\n    overlays = []\n\n    for yaw in yaw_degs:\n        calib = perturb_extrinsic(frame["calib"], yaw_deg=yaw)\n        uv, depth, valid = project_velo_to_image(raw_points, calib, image.shape)\n        members, _ = _box_membership(uv, frame["labels"])\n        n_projected = int(valid.sum())\n        n_box = int(members.sum())\n        rows.append({\n            "frame": frame_id,\n            "yaw_deg": yaw,\n            "input_points": int(len(raw_points)),\n            "finite_xyz_points": int(finite_xyz.sum()),\n            "inside_image_points": n_projected,\n            "inside_image_pct_of_finite": 100.0 * n_projected / max(1, int(finite_xyz.sum())),\n            "points_in_any_gt_2d_box": n_box,\n            "gt_box_pct_of_projected": 100.0 * n_box / max(1, n_projected),\n            "labeled_objects": len(frame["labels"]),\n        })\n        vis = overlay_points(image, uv, depth)\n        for obj in frame["labels"]:\n            x1, y1, x2, y2 = (int(round(v)) for v in obj.bbox)\n            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)\n            cv2.putText(vis, obj.type, (x1, max(16, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX,\n                        0.5, (0, 255, 0), 1, cv2.LINE_AA)\n        cv2.putText(vis, f"yaw drift = {yaw:.1f} deg | in FOV = {n_projected} | in GT boxes = {n_box}",\n                    (18, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)\n        overlays.append((yaw, vis))\n\n    out_dir.mkdir(parents=True, exist_ok=True)\n    results_dir = out_dir.parent\n    with (results_dir / "yaw_perturb_sweep.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n    baseline_idx = min(range(len(overlays)), key=lambda i: abs(overlays[i][0]))\n    baseline = overlays[baseline_idx][1]\n    cv2.imwrite(str(out_dir / f"topic_a_overlay_{frame_id}.png"), baseline)\n    worst_idx = max(range(len(overlays)), key=lambda i: abs(overlays[i][0]))\n    if worst_idx == baseline_idx and len(overlays) > 1:\n        worst_idx = len(overlays) - 1\n    yaw_fail, fail_img = overlays[worst_idx]\n    cv2.imwrite(str(out_dir / f"fail_01_yaw_{yaw_fail:g}deg_{frame_id}.png"), fail_img)\n\n    fig, ax = plt.subplots(figsize=(7.5, 4.5))\n    ax.plot([r["yaw_deg"] for r in rows], [r["inside_image_pct_of_finite"] for r in rows],\n            "o-", label="inside image / finite points")\n    ax.plot([r["yaw_deg"] for r in rows], [r["gt_box_pct_of_projected"] for r in rows],\n            "s-", label="inside labeled 2D boxes / projected")\n    ax.set(xlabel="Injected LiDAR yaw error (degrees)", ylabel="Points (%)",\n           title=f"Calibration yaw sweep — {frame_id}")\n    ax.grid(True, alpha=0.3)\n    ax.legend()\n    fig.tight_layout()\n    fig.savefig(out_dir / "yaw_perturb_sweep.png", dpi=160)\n    plt.close(fig)\n    return rows\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--data-root", default="data/synthetic")\n    parser.add_argument("--frame", default="000000")\n    parser.add_argument("--yaw-degs", nargs="+", type=float, default=[0, 0.5, 1, 2, 3])\n    parser.add_argument("--out-dir", type=Path, default=Path("results/figures"))\n    args = parser.parse_args()\n    rows = run(args.data_root, args.frame, args.yaw_degs, args.out_dir)\n    print("yaw_deg  inside_image_points  in_gt_boxes  fov_pct  gt_box_pct")\n    for row in rows:\n        print(f"{row[\'yaw_deg\']:>7.2f} {row[\'inside_image_points\']:>19} "\n              f"{row[\'points_in_any_gt_2d_box\']:>12} "\n              f"{row[\'inside_image_pct_of_finite\']:>8.3f} "\n              f"{row[\'gt_box_pct_of_projected\']:>11.4f}")\n    print(f"Saved CSV: {args.out_dir.parent / \'yaw_perturb_sweep.csv\'}")\n    print(f"Saved figures: {args.out_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', 'report/REPORT.md': '# Báo cáo Day 6: Độ nhạy projection LiDAR-camera với yaw drift\n\n- **Họ tên:** Nguyễn Văn Chiến\n- **MSSV:** 2A202602926\n- **Lớp:** AI20K\n- **Link repo:** https://github.com/nvchien19/K4-Track4-Day06-3D-From-Point-Clouds\n- **Topic:** A — LiDAR-camera projection QA\n- **Dataset:** `data/synthetic`, `data/kitti_mini`\n- **Các frame đã dùng:** `000000` (benchmark); `000000` synthetic và `000011`, `000012`, `000015`, `000019` KITTI (overlay demo)\n\n## 1. Claim\n\nTrên frame synthetic `000000`, yaw drift +3° làm tỷ lệ điểm chiếu nằm trong 2D GT boxes giảm ít nhất 5 điểm phần trăm so với calibration gốc, trong khi tỷ lệ điểm còn nằm trong FOV không giảm.\n\n## 2. Evidence\n\n| Yaw | Điểm trong ảnh | Điểm trong GT boxes | GT-box / projected |\n|---:|---:|---:|---:|\n| 0° | 3,910 (16.339%) | 1,378 | 35.243% |\n| 0.5° | 3,928 (16.415%) | 1,328 | 33.809% |\n| 1° | 3,935 (16.444%) | 1,296 | 32.935% |\n| 2° | 3,956 (16.532%) | 1,213 | 30.662% |\n| 3° | 3,996 (16.699%) | 1,183 | 29.605% |\n\nCSV: `results/yaw_perturb_sweep.csv`. Plot: `results/figures/yaw_perturb_sweep.png`. Overlay baseline: `results/figures/topic_a_overlay_000000.png`.\nBa overlay KITTI thêm: `results/figures/overlay_000011_*.png`, `overlay_000012_*.png`, `overlay_000015_*.png`, và `overlay_000019_*.png`.\n\n![Overlay baseline](../results/figures/topic_a_overlay_000000.png)\n\n## 3. Failure case\n\nỞ yaw drift +3°, điểm quanh vật thể không còn khớp với 2D box gốc; tỷ lệ GT-box giảm 5.64 điểm phần trăm (16.0% tương đối). Đây là lỗi **Geometry** do extrinsic bị perturb; thêm vào đó, chỉ số FOV là **Metric** yếu vì nó tăng nhẹ dù alignment xấu đi.\n\n![Failure at 3 degree yaw drift](../results/figures/fail_01_yaw_3deg_000000.png)\n\nVới xe thật, theo dõi thêm edge/box alignment và cảnh báo theo xu hướng qua nhiều frame; không dùng riêng tỷ lệ điểm trong ảnh.\n\n## 4. Khuyến nghị nếu triển khai thật\n\nTrong ADAS, kiểm tra alignment sau va chạm hoặc bảo dưỡng giá đỡ cảm biến. Theo dõi tỷ lệ điểm trong vùng vật thể/edge, số điểm hợp lệ và độ ổn định qua thời gian; xác nhận trên nhiều cảnh KITTI/log thật trước khi chọn ngưỡng. Bước này rẻ hơn chạy detector, nhưng box label trong thử nghiệm synthetic không thay cho đánh giá an toàn ngoài đường.\n\n## 5. Cách chạy lại\n\n```bash\npython -m starter.data_health --data-root data/synthetic\npython -m starter.projection --data-root data/synthetic --frame 000000\npython -m starter.projection --data-root data/kitti_mini --frame 000011\npython -m src.topic_a_calibration --data-root data/synthetic --frame 000000 --yaw-degs 0 0.5 1 2 3\n```\n\n## 6. Khai báo sử dụng AI\n\n| Công cụ | Dùng cho việc gì | Bạn đã kiểm chứng thế nào |\n|---|---|---|\n| OpenAI Codex | Đọc README, cài hai hàm projection, tạo benchmark/plot và soạn báo cáo | Chạy kiểm tra hai bộ dữ liệu, known-point `(10,0,0)` cho `z≈9.727`, `(u,v)≈(613.964,175.007)`, chạy benchmark hai lần và so sánh CSV trước khi nộp |\n'}
for relative, content in PAYLOAD.items():
    target = work / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')

def run_module(module, *args):
    subprocess.run([sys.executable, '-m', module, *map(str, args)], check=True, cwd=work)

def run_script(script, *args):
    subprocess.run([sys.executable, script, *map(str, args)], check=True, cwd=work)
print('Đã khôi phục projection, benchmark và báo cáo.')


## 2. Kiểm tra dữ liệu và hình học
Điểm LiDAR `(10,0,0)` phải chiếu gần `(614,175)`. NaN, Inf và điểm sau camera phải bị loại.

In [ ]:
run_script('tools/verify_data.py', '--data-root', 'data/kitti_mini')
run_script('tools/verify_data.py', '--data-root', 'data/nuscenes_mini_subset')
run_module('starter.data_health', '--data-root', 'data/synthetic')
import numpy as np
from starter.datasets import load_frame
from starter.projection import velo_to_cam, cam_to_image
fr = load_frame('data/synthetic', '000000')
cam = velo_to_cam(np.array([[10., 0., 0.]]), fr['calib'])
uv, depth, mask = cam_to_image(cam, fr['calib'].P2, fr['image'].shape)
assert mask.tolist() == [True]
np.testing.assert_allclose(depth, [9.72732108921415], atol=0.001)
np.testing.assert_allclose(uv[0], [613.9641486888877, 175.00653723101712], atol=0.01)
bad = np.array([[0,0,5], [np.nan,0,5], [0,0,np.inf], [0,0,-1]], float)
u, d, m = cam_to_image(bad, fr['calib'].P2, fr['image'].shape)
assert m.tolist() == [True, False, False, False]
assert np.isfinite(u).all() and np.isfinite(d).all()
print('PASS: known point và NaN/Inf/depth filtering', cam, uv)


## 3. Chạy overlay demo
Synthetic, bốn frame KITTI và một frame nuScenes.

In [ ]:
run_module('starter.projection', '--data-root', 'data/synthetic', '--frame', '000000')
for frame_id in ['000011', '000012', '000015', '000019']:
    run_module('starter.projection', '--data-root', 'data/kitti_mini', '--frame', frame_id)
run_module('starter.projection', '--data-root', 'data/nuscenes_mini_subset', '--frame', 'scene-0103_010')


## 4. Benchmark yaw 0°, 0.5°, 1°, 2°, 3°
Giữ nguyên frame, điểm và box. Metric box đếm điểm trong hợp các box 2D (không đếm trùng), nên có thể chứa cả điểm nền. Đây là proxy alignment, không phải recall detector. Hai lần chạy phải cho CSV giống hệt nhau.

In [ ]:
import hashlib, csv
args = ['--data-root', 'data/synthetic', '--frame', '000000', '--yaw-degs', '0', '0.5', '1', '2', '3']
run_module('src.topic_a_calibration', *args)
csv_path = work / 'results/yaw_perturb_sweep.csv'
first = hashlib.sha256(csv_path.read_bytes()).hexdigest()
run_module('src.topic_a_calibration', *args)
assert first == hashlib.sha256(csv_path.read_bytes()).hexdigest(), 'CSV không tái lập được.'
with csv_path.open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
print('PASS: CSV tái lập được. SHA256:', first)
from IPython.display import display, Markdown, Image
header = '| Yaw | Điểm trong ảnh | Điểm trong GT boxes | GT-box/projected |\n|---:|---:|---:|---:|'
table = header + '\n' + '\n'.join(
    f"| {float(r['yaw_deg']):g}° | {int(r['inside_image_points']):,} ({float(r['inside_image_pct_of_finite']):.3f}%) | {int(r['points_in_any_gt_2d_box']):,} | {float(r['gt_box_pct_of_projected']):.3f}% |" for r in rows)
display(Markdown(table))
for filename in ['yaw_perturb_sweep.png', 'topic_a_overlay_000000.png', 'fail_01_yaw_3deg_000000.png']:
    display(Image(filename=str(work / 'results/figures' / filename)))


## 5. Báo cáo từ kết quả của phiên này
Box alignment có thể giảm dù FOV tăng: xem hình để tự xác nhận sai lệch ở cột, xe và người. Kết luận chỉ áp dụng cho frame synthetic này.

In [ ]:
import re, platform, json
base, last = rows[0], rows[-1]
b0 = float(base['gt_box_pct_of_projected'])
b3 = float(last['gt_box_pct_of_projected'])
drop = b0 - b3
relative_drop = 100 * drop / b0
fov0 = float(base['inside_image_pct_of_finite'])
fov3 = float(last['inside_image_pct_of_finite'])
report = PAYLOAD['report/REPORT.md']
report = re.sub(r'\| Yaw \|.*?(?=\n\nCSV:)', table, report, flags=re.S)
report = report.replace('giảm ít nhất 5 điểm phần trăm', f'giảm {drop:.2f} điểm phần trăm')
report = report.replace('trong khi tỷ lệ điểm còn nằm trong FOV không giảm.', f'trong khi tỷ lệ điểm còn nằm trong FOV đổi từ {fov0:.3f}% thành {fov3:.3f}%.')
report = report.replace('5.64 điểm phần trăm (16.0% tương đối)', f'{drop:.2f} điểm phần trăm ({relative_drop:.1f}% tương đối)')
report += '\n\n### Thông tin lần chạy Colab\n\n' + f'Python {platform.python_version()}; NumPy {np.__version__}; base commit `{BASE_COMMIT}`; CSV SHA256 `{first}`.\n'
(work / 'report/REPORT.md').write_text(report, encoding='utf-8')
metadata = {'platform': platform.platform(), 'python': platform.python_version(), 'numpy': np.__version__, 'base_commit': BASE_COMMIT, 'csv_sha256': first, 'yaw_degs': [0, 0.5, 1, 2, 3]}
(work / 'results/colab_run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
run_script('tools/check_submission.py')
display(Markdown(report))


## 6. Tải code, CSV, ảnh và báo cáo về máy
Chép các thư mục trong ZIP vào repo cục bộ; xem lại báo cáo và tên lớp trước khi nộp. ZIP chỉ là gói tải xuống, không commit ZIP vào repo.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path('/content/NguyenVanChien-2A202602926-TopicA-results.zip')
with ZipFile(archive, 'w', ZIP_DEFLATED) as z:
    for folder in ['src', 'results', 'report']:
        for path in (work / folder).rglob('*'):
            if path.is_file() and '__pycache__' not in path.parts and path.suffix != '.pyc':
                z.write(path, path.relative_to(work))
    z.write(work / 'starter/projection.py', 'starter/projection.py')
print('Tải xuống:', archive.name)
files.download(str(archive))
